# Object Detection Project

## 1. Settings

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/HoangDuonng1359/object-detection"
PROJECT_DIR = Path("/kaggle/working/object-detection")

IMAGE_SIZE = 512
EPOCHS = 70
BATCH_SIZE = 4
NUM_WORKERS = 1
LR = 1e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
USE_PRETRAINED_BACKBONE = True
FREEZE_BACKBONE_STEM = False
OVERSAMPLE_CLASSES = ["chair"]
OVERSAMPLE_FACTOR = 2.0

# Set these to small values for a quick smoke test. Use 0 for full train/validation.
MAX_TRAIN_BATCHES = 0
MAX_VAL_BATCHES = 0

CONF_THRESHOLD = 0.25
# Tuned from validation: chair has low recall, car has room to trade precision for recall.
CLASS_THRESHOLDS = {
    "person": 0.25,
    "car": 0.20,
    "dog": 0.25,
    "cat": 0.25,
    "chair": 0.10,
}
CLASS_THRESHOLDS_TEXT = ",".join(f"{name}={value}" for name, value in CLASS_THRESHOLDS.items())
NMS_THRESHOLD = 0.35
MAX_DETECTIONS = 100

RUN_THRESHOLD_SWEEP = True
CLASS_THRESHOLD_CONFIGS = [
    {
        "name": "balanced",
        "default": 0.25,
        "thresholds": CLASS_THRESHOLDS,
    },
    {
        "name": "more_recall",
        "default": 0.20,
        "thresholds": {"person": 0.20, "car": 0.15, "dog": 0.20, "cat": 0.20, "chair": 0.08},
    },
    {
        "name": "clean",
        "default": 0.30,
        "thresholds": {"person": 0.30, "car": 0.25, "dog": 0.25, "cat": 0.30, "chair": 0.15},
    },
    {
        "name": "map_oriented",
        "default": 0.15,
        "thresholds": {"person": 0.15, "car": 0.15, "dog": 0.15, "cat": 0.15, "chair": 0.08},
    },
]
NMS_VALUES = [0.35, 0.45, 0.50]

# Optional hidden/test image directory. Leave as None if you only want val predictions.
TEST_IMAGE_DIR = None
FINAL_PREDICTIONS = Path("/kaggle/working/predictions.json")


## 2. Clone Code And Install Requirements

In [2]:
import os
import subprocess
import sys

if not PROJECT_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
        raise ValueError("Edit REPO_URL in the settings cell before cloning.")
    clone_cmd = ["git", "clone"]
    clone_cmd += [REPO_URL, str(PROJECT_DIR)]
    subprocess.run(clone_cmd, check=True)
else:
    print(f"Project directory already exists: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

requirements = PROJECT_DIR / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

import torch
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())


Cloning into '/kaggle/working/object-detection'...


cwd: /kaggle/working/object-detection
python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.10.0+cu128
cuda_available: True


## 3. Locate Kaggle Input Dataset

In [3]:
PUBLIC_DIR = Path("/kaggle/input/datasets/duong1589/object-detection/public")
TRAIN_JSON = PUBLIC_DIR / "annotations" / "train.json"
VAL_JSON = PUBLIC_DIR / "annotations" / "val.json"
TRAIN_IMAGES = PUBLIC_DIR / "train" / "images"
VAL_IMAGES = PUBLIC_DIR / "val" / "images"
EVALUATOR = PUBLIC_DIR / "tools" / "evaluate_predictions.py"

print("PUBLIC_DIR:", PUBLIC_DIR)
print("TRAIN_JSON:", TRAIN_JSON)
print("VAL_JSON:", VAL_JSON)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("VAL_IMAGES:", VAL_IMAGES)
print("EVALUATOR:", EVALUATOR, EVALUATOR.exists())


PUBLIC_DIR: /kaggle/input/datasets/duong1589/object-detection/public
TRAIN_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json
VAL_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json
TRAIN_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/train/images
VAL_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/val/images
EVALUATOR: /kaggle/input/datasets/duong1589/object-detection/public/tools/evaluate_predictions.py True


## 4. Sanity Check Dataset And Code

In [4]:
import json

for split_name, json_path in [("train", TRAIN_JSON), ("val", VAL_JSON)]:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    print(split_name, "images=", len(data["images"]), "annotations=", len(data["annotations"]))

subprocess.run([sys.executable, "-m", "py_compile", "train.py", "predict.py"], check=True)
subprocess.run([
    sys.executable, "-m", "utils.check_dataset",
    "--annotations", str(TRAIN_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--image_size", str(IMAGE_SIZE),
    "--batch_size", "2",
    "--train",
], check=True)


train images= 7500 annotations= 10642
val images= 1500 annotations= 2021
dataset_size=7500
classes=['person', 'car', 'dog', 'cat', 'chair']
sample_image_shape=(3, 512, 512)
sample_image_id=img_000c52c6d12f.jpg
sample_boxes=(1, 4)
batch_image_shape=(2, 3, 512, 512)
batch_box_counts=[0, 2]


CompletedProcess(args=['/usr/bin/python3', '-m', 'utils.check_dataset', '--annotations', '/kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json', '--image_dir', '/kaggle/input/datasets/duong1589/object-detection/public/train/images', '--image_size', '512', '--batch_size', '2', '--train'], returncode=0)

## 5. Train

In [5]:
from IPython.display import display
import pandas as pd

train_cmd = [
    sys.executable, "train.py",
    "--train_data", str(TRAIN_JSON),
    "--val_data", str(VAL_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--val_image_dir", str(VAL_IMAGES),
    "--checkpoint_dir", str(PROJECT_DIR / "models"),
    "--image_size", str(IMAGE_SIZE),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--lr", str(LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--class_thresholds", CLASS_THRESHOLDS_TEXT,
    "--nms_threshold", str(NMS_THRESHOLD),
]
if USE_AMP:
    train_cmd.append("--amp")
if USE_PRETRAINED_BACKBONE:
    train_cmd.append("--pretrained_backbone")
if FREEZE_BACKBONE_STEM:
    train_cmd.append("--freeze_backbone_stem")
if OVERSAMPLE_CLASSES and OVERSAMPLE_FACTOR > 1.0:
    train_cmd += ["--oversample_classes", *OVERSAMPLE_CLASSES]
    train_cmd += ["--oversample_factor", str(OVERSAMPLE_FACTOR)]
if MAX_TRAIN_BATCHES > 0:
    train_cmd += ["--max_train_batches", str(MAX_TRAIN_BATCHES)]
if MAX_VAL_BATCHES > 0:
    train_cmd += ["--max_val_batches", str(MAX_VAL_BATCHES)]

print("Train command:")
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True)

history_files = sorted((PROJECT_DIR / "models").glob("train_history_*.csv"), key=lambda item: item.stat().st_mtime)
if history_files:
    HISTORY_CSV = history_files[-1]
    history_df = pd.read_csv(HISTORY_CSV)
    print("History CSV:", HISTORY_CSV)
    display_columns = [
        "epoch", "lr", "train_loss", "val_loss", "val_map50", "best_map50",
        "train_box_loss", "train_objectness_loss", "train_class_loss",
        "val_box_loss", "val_objectness_loss", "val_class_loss",
    ]
    display(history_df[[col for col in display_columns if col in history_df.columns]].tail(10))
else:
    HISTORY_CSV = None
    print("No train history CSV found.")


Train command:
/usr/bin/python3 train.py --train_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json --val_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json --image_dir /kaggle/input/datasets/duong1589/object-detection/public/train/images --val_image_dir /kaggle/input/datasets/duong1589/object-detection/public/val/images --checkpoint_dir /kaggle/working/object-detection/models --image_size 512 --epochs 70 --batch_size 8 --num_workers 1 --lr 0.0001 --weight_decay 0.0001 --conf_threshold 0.25 --class_thresholds person=0.25,car=0.2,dog=0.25,cat=0.25,chair=0.1 --nms_threshold 0.35 --amp --pretrained_backbone --oversample_classes chair --oversample_factor 2.0
device=cuda amp=True gpu_count=2
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 165MB/s]


using DataParallel on 2 GPUs
class_thresholds: person=0.250, car=0.200, dog=0.250, cat=0.250, chair=0.100


epoch=1/70 lr=6.66667e-05 train_loss=3.0021 val_loss=2.4303 val_map50=0.3674 best_map50=0.3674


epoch=2/70 lr=0.0001 train_loss=2.3940 val_loss=2.2413 val_map50=0.3210 best_map50=0.3674


...

epoch=14/70 lr=9.34953e-05 train_loss=1.3535 val_loss=1.7068 val_map50=0.6000 best_map50=0.6074


epoch=15/70 lr=9.22916e-05 train_loss=1.3066 val_loss=1.6789 val_map50=0.6121 best_map50=0.6121


epoch=16/70 lr=9.09949e-05 train_loss=1.2906 val_loss=1.7033 val_map50=0.6042 best_map50=0.6121


train:  15%|█▍        | 138/938 [03:23<5:49:47, 26.23s/it, box=0.2080, cls=0.0837, loss=1.2749, obj=0.1932]

CalledProcessError: Command '['/usr/bin/python3', 'train.py', '--train_data', '/kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json', '--val_data', '/kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json', '--image_dir', '/kaggle/input/datasets/duong1589/object-detection/public/train/images', '--val_image_dir', '/kaggle/input/datasets/duong1589/object-detection/public/val/images', '--checkpoint_dir', '/kaggle/working/object-detection/models', '--image_size', '512', '--epochs', '70', '--batch_size', '8', '--num_workers', '1', '--lr', '0.0001', '--weight_decay', '0.0001', '--conf_threshold', '0.25', '--class_thresholds', 'person=0.25,car=0.2,dog=0.25,cat=0.25,chair=0.1', '--nms_threshold', '0.35', '--amp', '--pretrained_backbone', '--oversample_classes', 'chair', '--oversample_factor', '2.0']' died with <Signals.SIGKILL: 9>.

## 6. Predict On Validation And Evaluate

In [ ]:
VAL_PREDICTIONS = PROJECT_DIR / "val_predictions.json"
predict_val_cmd = [
    sys.executable, "predict.py",
    "--image_dir", str(VAL_IMAGES),
    "--output", str(VAL_PREDICTIONS),
    "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
    "--batch_size", str(BATCH_SIZE),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--class_thresholds", CLASS_THRESHOLDS_TEXT,
    "--nms_threshold", str(NMS_THRESHOLD),
    "--max_detections", str(MAX_DETECTIONS),
]
print("Predict validation command:")
print(" ".join(predict_val_cmd))
subprocess.run(predict_val_cmd, check=True)

val_predictions_data = json.loads(VAL_PREDICTIONS.read_text(encoding="utf-8"))
num_pred_boxes = sum(len(item["boxes"]) for item in val_predictions_data)
print(f"Validation predictions: images={len(val_predictions_data)} boxes={num_pred_boxes}")
print("Prediction preview:")
print(json.dumps(val_predictions_data[:3], ensure_ascii=False, indent=2))

if EVALUATOR.exists():
    VAL_SCORE = PROJECT_DIR / "val_score.json"
    subprocess.run([
        sys.executable, str(EVALUATOR),
        "--ground_truth", str(VAL_JSON),
        "--predictions", str(VAL_PREDICTIONS),
        "--output", str(VAL_SCORE),
    ], check=True)
    score = json.loads(VAL_SCORE.read_text(encoding="utf-8"))
    print("Validation score:")
    print(json.dumps(score, ensure_ascii=False, indent=2))
    if "per_class" in score:
        per_class_df = pd.DataFrame.from_dict(score["per_class"], orient="index")
        display(per_class_df)
else:
    VAL_SCORE = None
    print("Evaluator not found, skipped local validation scoring.")


## 7. Threshold Sweep On Validation


In [ ]:
import json
import pandas as pd

BEST_CHECKPOINT = PROJECT_DIR / "models" / "best.pth"

def thresholds_to_text(thresholds):
    return ",".join(f"{name}={value}" for name, value in thresholds.items())

if RUN_THRESHOLD_SWEEP:
    if not (VAL_JSON.exists() and EVALUATOR.exists()):
        raise FileNotFoundError("Need VAL_JSON and EVALUATOR for threshold sweep.")
    rows = []
    sweep_dir = PROJECT_DIR / "threshold_sweep"
    sweep_dir.mkdir(parents=True, exist_ok=True)
    for config in CLASS_THRESHOLD_CONFIGS:
        config_name = config["name"]
        default_conf = config["default"]
        thresholds = config["thresholds"]
        thresholds_text = thresholds_to_text(thresholds)
        safe_thresholds = thresholds_text.replace(",", "_").replace("=", "-")
        for nms in NMS_VALUES:
            sweep_pred = sweep_dir / f"predictions_{config_name}_nms{nms:.2f}_{safe_thresholds}.json"
            sweep_score = sweep_dir / f"score_{config_name}_nms{nms:.2f}_{safe_thresholds}.json"
            print(f"Running config={config_name} default={default_conf:.2f} thresholds={thresholds_text} nms={nms:.2f}")
            !{sys.executable} predict.py --image_dir "{VAL_IMAGES}" --output "{sweep_pred}" --checkpoint "{BEST_CHECKPOINT}" --batch_size {BATCH_SIZE} --conf_threshold {default_conf} --class_thresholds "{thresholds_text}" --nms_threshold {nms} --max_detections {MAX_DETECTIONS}
            !{sys.executable} "{EVALUATOR}" --ground_truth "{VAL_JSON}" --predictions "{sweep_pred}" --output "{sweep_score}"
            score = json.loads(sweep_score.read_text(encoding="utf-8"))
            per_class = score.get("per_class", {})
            chair = per_class.get("chair", {})
            rows.append({
                "config": config_name,
                "thresholds": thresholds_text,
                "default_conf": default_conf,
                "nms": nms,
                "map50": score.get("mAP@0.5", 0.0),
                "performance_points": score.get("performance_points", 0),
                "precision": score.get("micro_precision", 0.0),
                "recall": score.get("micro_recall", 0.0),
                "num_predictions": score.get("num_predictions", 0),
                "chair_ap": chair.get("ap", 0.0),
                "chair_precision": chair.get("precision", 0.0),
                "chair_recall": chair.get("recall", 0.0),
            })
    sweep_df = pd.DataFrame(rows).sort_values(["map50", "precision"], ascending=[False, False])
    sweep_csv = sweep_dir / "threshold_sweep_results.csv"
    sweep_df.to_csv(sweep_csv, index=False)
    print("Threshold sweep results:", sweep_csv)
    display(sweep_df)
else:
    print("Threshold sweep disabled. Set RUN_THRESHOLD_SWEEP = True to run it.")


## 8. Predict On Hidden/Test Images If Available


In [ ]:
if TEST_IMAGE_DIR is None:
    print("TEST_IMAGE_DIR is None. Skipped final prediction.")
else:
    test_dir = Path(TEST_IMAGE_DIR)
    final_cmd = [
        sys.executable, "predict.py",
        "--image_dir", str(test_dir),
        "--output", str(FINAL_PREDICTIONS),
        "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
        "--batch_size", str(BATCH_SIZE),
        "--conf_threshold", str(CONF_THRESHOLD),
        "--class_thresholds", CLASS_THRESHOLDS_TEXT,
        "--nms_threshold", str(NMS_THRESHOLD),
    ]
    print("Final predict command:")
    print(" ".join(final_cmd))
    subprocess.run(final_cmd, check=True)
    final_data = json.loads(FINAL_PREDICTIONS.read_text(encoding="utf-8"))
    print(f"Final predictions: images={len(final_data)} boxes={sum(len(item['boxes']) for item in final_data)}")
    print("Final prediction preview:")
    print(json.dumps(final_data[:3], ensure_ascii=False, indent=2))
    print("Final predictions path:", FINAL_PREDICTIONS)


## 9. Collect Artifacts


In [ ]:
import shutil

ARTIFACT_DIR = Path("/kaggle/working/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    PROJECT_DIR / "models" / "best.pth",
    PROJECT_DIR / "models" / "last.pth",
    VAL_PREDICTIONS,
    PROJECT_DIR / "val_score.json",
]
files_to_copy += sorted((PROJECT_DIR / "models").glob("train_history_*.csv"))
sweep_results = PROJECT_DIR / "threshold_sweep" / "threshold_sweep_results.csv"
if sweep_results.exists():
    files_to_copy.append(sweep_results)
if FINAL_PREDICTIONS.exists():
    files_to_copy.append(FINAL_PREDICTIONS)

copied = []
for src in files_to_copy:
    if src.exists():
        dst = ARTIFACT_DIR / src.name
        shutil.copy2(src, dst)
        copied.append(dst)

zip_path = shutil.make_archive("/kaggle/working/object_detection_artifacts", "zip", ARTIFACT_DIR)
print("Artifacts directory:", ARTIFACT_DIR)
print("Artifacts zip:", zip_path)
print("Files:")
artifact_rows = []
for path in sorted(ARTIFACT_DIR.iterdir()):
    artifact_rows.append({"file": path.name, "size_mb": round(path.stat().st_size / (1024 * 1024), 3)})
display(pd.DataFrame(artifact_rows))


## 10. Summary Shown In Notebook


In [ ]:
summary = {
    "best_checkpoint": str(PROJECT_DIR / "models" / "best.pth"),
    "history_csv": str(HISTORY_CSV) if HISTORY_CSV is not None else None,
    "val_predictions": str(VAL_PREDICTIONS),
    "val_score": str(PROJECT_DIR / "val_score.json"),
    "threshold_sweep_results": str(PROJECT_DIR / "threshold_sweep" / "threshold_sweep_results.csv"),
    "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

if HISTORY_CSV is not None:
    history_df = pd.read_csv(HISTORY_CSV)
    print("Final history row:")
    display(history_df.tail(1))

score_path = PROJECT_DIR / "val_score.json"
if score_path.exists():
    score = json.loads(score_path.read_text(encoding="utf-8"))
    print(f"mAP@0.5={score.get('mAP@0.5')} performance_points={score.get('performance_points')}")
